# Sprint 2: StatsBomb data -> CSVs -> model-ready tables

**Project:** freeze-frame xG, comparing a set model with a graph model.

This notebook does two jobs.

| Stage | Reads | Writes |
|---|---|---|
| 1. Download and convert | StatsBomb open data JSON on GitHub (`hudl/open-data`) | `data/raw/summary.csv`, `matches.csv`, `shots.csv`, `freeze_frames.csv` |
| 2. Process | `data/raw/shots.csv`, `data/raw/freeze_frames.csv` | `data/processed/shot_table.csv`, `player_table.csv` |

Scope: **men's** matches from the top-5 European leagues (Premier League, La Liga, Serie A, Bundesliga, Ligue 1).

Run it with `pip install requests pandas numpy tqdm` and then *Run All*. The first run downloads thousands of event files, so it can take a while. Downloaded data is cached in `data/raw/shots/` and `data/raw/matches/`, so re-running is quick and an interrupted run resumes where it stopped.

**Data licence:** StatsBomb open data requires attribution. See https://github.com/hudl/open-data

## 0. Setup

In [1]:
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

BASE = "https://raw.githubusercontent.com/hudl/open-data/master/data"

RAW = Path("data/raw")
PROCESSED = Path("data/processed")

# Create the folders if they do not exist (parents=True also creates "data/").
for folder in (RAW, RAW / "shots", RAW / "matches", PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8

# Set to a small number (e.g. 50) for a quick dry run on a random sample of matches.
# Leave as None for the full dataset.
DEBUG_MAX_MATCHES = None

TOP5 = {
    "Premier League": ("England", "Premier League"),
    "La Liga": ("Spain", "La Liga"),
    "Serie A": ("Italy", "Serie A"),
    "Bundesliga": ("Germany", "1. Bundesliga"),
    "Ligue 1": ("France", "Ligue 1"),
}

# Pitch geometry (StatsBomb coordinates: 120 x 80, every shot attacks towards x = 120).
PITCH_L, PITCH_W = 120.0, 80.0
GOAL_X, GOAL_Y = 120.0, 40.0
POST_LOW, POST_HIGH = 36.0, 44.0

# Collect pass/fail results of all checks so we can summarise them at the end.
CHECKS = []

def check(name, condition, detail=""):
    CHECKS.append((name, bool(condition), detail))
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")

# Stage 1: download the JSON files and convert to CSV

## 1.1 Download helpers
Requests retry automatically on temporary server errors, and files are written to a temporary name first so an interrupted download cannot leave a corrupt cache file.

In [2]:
session = requests.Session()
retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=MAX_WORKERS * 2))


def write_atomic(path, text):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def get_json(url):
    """Download JSON; return None if the file does not exist (404)."""
    response = session.get(url, timeout=60)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()


def cached_json(url, path):
    """Load JSON from the local cache, or download it and cache it."""
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    data = get_json(url)
    if data is not None:
        write_atomic(path, json.dumps(data))
    return data

## 1.2 Competitions: men's top-5 league seasons

In [3]:
comp_df = pd.DataFrame(cached_json(f"{BASE}/competitions.json", RAW / "competitions.json"))

is_male = comp_df["competition_gender"] == "male"
if "competition_youth" in comp_df.columns:
    not_youth = ~comp_df["competition_youth"].fillna(False).astype(bool)
else:
    not_youth = pd.Series(True, index=comp_df.index)

parts = []
for league, (country, name) in TOP5.items():
    sel = comp_df[
        is_male & not_youth
        & (comp_df["country_name"] == country)
        & (comp_df["competition_name"] == name)
    ].copy()
    if sel.empty:
        print(f"WARNING: no rows found for {league} ({country} / {name})")
        continue
    sel["league"] = league
    parts.append(sel)

if not parts:
    raise ValueError("No matching competitions found.")

league_seasons = pd.concat(parts, ignore_index=True)
print(f"{len(league_seasons)} men's top-5 league seasons found")
league_seasons[["league", "season_name"]].groupby("league").size()

27 men's top-5 league seasons found


league
Bundesliga         2
La Liga           18
Ligue 1            3
Premier League     2
Serie A            2
dtype: int64

## 1.3 Matches and season completeness
`season_complete` is `True` only if the season is a full double round-robin (every team plays every other team once at home and once away). Incomplete seasons are mostly club-focused subsets, for example one team's matches.

In [4]:
match_rows = []

for _, c in tqdm(league_seasons.iterrows(), total=len(league_seasons), desc="Match lists"):
    cid, sid = int(c["competition_id"]), int(c["season_id"])

    matches = cached_json(
        f"{BASE}/matches/{cid}/{sid}.json",
        RAW / "matches" / f"{cid}_{sid}.json",
    ) or []

    pairs = [(m["home_team"]["home_team_name"], m["away_team"]["away_team_name"]) for m in matches]
    teams = {t for pair in pairs for t in pair}
    n_teams = len(teams)
    pair_counts = Counter(pairs)
    expected_pairs = {(h, a) for h in teams for a in teams if h != a}

    complete = (
        n_teams >= 2
        and len(matches) == n_teams * (n_teams - 1)
        and set(pair_counts) == expected_pairs
        and all(n == 1 for n in pair_counts.values())
    )

    for m in matches:
        match_rows.append({
            "match_id": m["match_id"],
            "league": c["league"],
            "season": c["season_name"],
            "match_date": m.get("match_date"),
            "home_team": m["home_team"]["home_team_name"],
            "away_team": m["away_team"]["away_team_name"],
            "season_complete": complete,
            "match_status": m.get("match_status"),
        })

matches_df = pd.DataFrame(match_rows)
matches_df = (
    matches_df[matches_df["match_status"] == "available"]
    .drop(columns="match_status")
    .reset_index(drop=True)
)

if DEBUG_MAX_MATCHES:
    matches_df = matches_df.sample(DEBUG_MAX_MATCHES, random_state=0).reset_index(drop=True)
    print(f"DEBUG MODE: using a random sample of {len(matches_df)} matches")

print(f"{len(matches_df)} matches to download")
matches_df.head()

Match lists:   0%|          | 0/27 [00:00<?, ?it/s]

2169 matches to download


,match_id,league,season,match_date,home_team,away_team,season_complete
0,3754217,Premier League,2015/2016,2015-09-19,Chelsea,Arsenal,True
1,3754117,Premier League,2015/2016,2015-12-13,Aston Villa,Arsenal,True
2,3754296,Premier League,2015/2016,2015-12-21,Arsenal,Manchester City,True
3,3753983,Premier League,2015/2016,2015-10-31,Swansea City,Arsenal,True
4,3754160,Premier League,2015/2016,2015-12-05,Arsenal,Sunderland,True


## 1.4 Events -> shots with freeze frames
For each match we download the full event file, keep only the `Shot` events, and cache just those. The freeze frame is the list of visible players at the moment of the shot.

In [5]:
def get_shots(match_id):
    """Extract the shots from one match (from the local cache if available)."""
    cache = RAW / "shots" / f"{match_id}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))

    events = get_json(f"{BASE}/events/{match_id}.json") or []
    shots = []

    for e in events:
        if e.get("type", {}).get("name") != "Shot":
            continue

        s = e.get("shot", {})
        location = e.get("location") or [None, None]
        outcome_name = s.get("outcome", {}).get("name")

        shots.append({
            "shot_id": e["id"],
            "match_id": match_id,
            "minute": e.get("minute"),
            "second": e.get("second"),
            "period": e.get("period"),
            "team": e.get("team", {}).get("name"),
            "player": e.get("player", {}).get("name"),
            "play_pattern": e.get("play_pattern", {}).get("name"),
            "under_pressure": e.get("under_pressure", False),
            "location_x": location[0] if len(location) > 0 else None,
            "location_y": location[1] if len(location) > 1 else None,
            "end_location": s.get("end_location"),
            "shot_type": s.get("type", {}).get("name"),
            "body_part": s.get("body_part", {}).get("name"),
            "technique": s.get("technique", {}).get("name"),
            "first_time": s.get("first_time", False),
            "outcome": outcome_name,
            "is_goal": outcome_name == "Goal",
            # Benchmark only: never use as a model feature.
            "statsbomb_xg": s.get("statsbomb_xg"),
            # Positions of visible players at the moment of the shot.
            "freeze_frame": s.get("freeze_frame"),
        })

    write_atomic(cache, json.dumps(shots))
    return shots


match_ids = matches_df["match_id"].tolist()
all_shots = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for shots in tqdm(executor.map(get_shots, match_ids), total=len(match_ids), desc="Downloading events"):
        all_shots.extend(shots)

print(f"{len(all_shots):,} shots extracted")

54,153 shots extracted


## 1.5 Build the four tables
Excel and Google Sheets cannot hold a list inside a cell, so the nested freeze frame becomes its own table with one row per visible player. `shot_id` links `shots.csv` to `freeze_frames.csv`, and `match_id` links `shots.csv` to `matches.csv`.

In [6]:
shots_df = pd.DataFrame(all_shots)

# --- end_location is [x, y] or [x, y, z] -> three columns ---------------------
def split_end_location(loc):
    if not isinstance(loc, (list, tuple)):
        return (None, None, None)
    loc = list(loc) + [None] * (3 - len(loc))
    return tuple(loc[:3])

end_xyz = shots_df["end_location"].apply(split_end_location)
shots_df["end_x"] = end_xyz.str[0]
shots_df["end_y"] = end_xyz.str[1]
shots_df["end_z"] = end_xyz.str[2]

# --- freeze frames -> long table + per-shot summary -------------------------------
ff_rows = []
ff_stats = []

for shot_id, ff in zip(shots_df["shot_id"], shots_df["freeze_frame"]):
    if not isinstance(ff, list) or len(ff) == 0:
        ff_stats.append((0, 0, 0, False))
        continue

    n_team = 0
    opp_keeper = False

    for p in ff:
        loc = p.get("location") or [None, None]
        position = p.get("position", {}).get("name")
        teammate = bool(p.get("teammate", False))
        is_keeper = position == "Goalkeeper"

        n_team += teammate
        opp_keeper = opp_keeper or (is_keeper and not teammate)

        ff_rows.append({
            "shot_id": shot_id,
            "x": loc[0],
            "y": loc[1],
            "teammate": teammate,
            "player_name": p.get("player", {}).get("name"),
            "position": position,
            "is_keeper": is_keeper,
        })

    ff_stats.append((len(ff), n_team, len(ff) - n_team, opp_keeper))

shots_df[["ff_n_players", "ff_n_teammates", "ff_n_opponents", "ff_has_keeper"]] = pd.DataFrame(
    ff_stats, index=shots_df.index
)
shots_df["has_freeze_frame"] = shots_df["ff_n_players"] > 0

# --- attach match information -----------------------------------------------------
shots_df = shots_df.merge(
    matches_df[["match_id", "league", "season", "match_date", "season_complete"]],
    on="match_id", how="left",
)

SHOT_COLUMNS = [
    "shot_id", "match_id", "minute", "second", "period", "team", "player",
    "play_pattern", "under_pressure", "location_x", "location_y",
    "shot_type", "body_part", "technique", "first_time", "outcome", "is_goal",
    "statsbomb_xg", "league", "season", "match_date", "season_complete",
    "has_freeze_frame", "end_x", "end_y", "end_z",
    "ff_n_players", "ff_n_teammates", "ff_n_opponents", "ff_has_keeper",
]
shots_out = shots_df[SHOT_COLUMNS].copy()

freeze_frames_out = pd.DataFrame(
    ff_rows, columns=["shot_id", "x", "y", "teammate", "player_name", "position", "is_keeper"]
)

summary_out = (
    shots_out
    .assign(is_penalty=shots_out["shot_type"] == "Penalty")
    .groupby(["league", "season", "season_complete"])
    .agg(
        matches=("match_id", "nunique"),
        shots=("shot_id", "size"),
        penalties=("is_penalty", "sum"),
        with_freeze_frame=("has_freeze_frame", "sum"),
        goals=("is_goal", "sum"),
    )
    .reset_index()
)

matches_out = matches_df.copy()

print(f"summary:       {summary_out.shape}")
print(f"matches:       {matches_out.shape}")
print(f"shots:         {shots_out.shape}")
print(f"freeze_frames: {freeze_frames_out.shape}")

summary:       (27, 8)
matches:       (2169, 7)
shots:         (54153, 30)
freeze_frames: (685059, 7)


## 1.6 Checks and verification (before saving)
These catch broken downloads, mismatched tables and obviously wrong values.

In [7]:
CHECKS.clear()

# Structure
check("All 5 leagues found", league_seasons["league"].nunique() == 5,
      f"({league_seasons['league'].nunique()} found)")
check("match_id unique in matches", matches_out["match_id"].is_unique)
check("shot_id unique in shots", shots_out["shot_id"].is_unique)
check("Every shot belongs to a known match", shots_out["match_id"].isin(matches_out["match_id"]).all())
check("Every freeze-frame row links to a shot", freeze_frames_out["shot_id"].isin(shots_out["shot_id"]).all())

# Freeze-frame consistency: row counts per shot equal the summary column
ff_counts = freeze_frames_out.groupby("shot_id").size()
expected = shots_out.set_index("shot_id")["ff_n_players"]
expected = expected[expected > 0]
check("Freeze-frame row counts match ff_n_players",
      ff_counts.reindex(expected.index).eq(expected).all())
check("Summary totals equal the shots table", summary_out["shots"].sum() == len(shots_out))

# Values
key_cols = ["shot_id", "match_id", "location_x", "location_y", "is_goal", "shot_type", "body_part"]
check("No missing values in key shot columns", shots_out[key_cols].notna().all().all())
check("is_goal agrees with outcome == 'Goal'", (shots_out["is_goal"] == (shots_out["outcome"] == "Goal")).all())
check("statsbomb_xg within [0, 1]", shots_out["statsbomb_xg"].between(0, 1).all())

# StatsBomb occasionally records locations a fraction of a yard outside the pitch,
# so allow a 1-yard tolerance and report how many are strictly outside.
in_pitch = shots_out["location_x"].between(-1, PITCH_L + 1) & shots_out["location_y"].between(-1, PITCH_W + 1)
strict = shots_out["location_x"].between(0, PITCH_L) & shots_out["location_y"].between(0, PITCH_W)
check("Shot locations on the pitch (1-yard tolerance)", in_pitch.all(),
      f"({(~strict).sum()} strictly outside 0-120 x 0-80)")
check("Freeze-frame locations on the pitch",
      freeze_frames_out["x"].between(0, PITCH_L).all() and freeze_frames_out["y"].between(0, PITCH_W).all())
check("Shots attack towards x = 120 (median x > 90)", shots_out["location_x"].median() > 90,
      f"(median x = {shots_out['location_x'].median():.1f})")

nonpen = shots_out[shots_out["shot_type"] != "Penalty"]
rate = nonpen["is_goal"].mean()
check("Non-penalty goal rate plausible (5-15%)", 0.05 < rate < 0.15, f"({rate:.3f})")

print()
print(f"{sum(ok for _, ok, _ in CHECKS)}/{len(CHECKS)} checks passed")

PASS  All 5 leagues found  (5 found)
PASS  match_id unique in matches  
PASS  shot_id unique in shots  
PASS  Every shot belongs to a known match  
PASS  Every freeze-frame row links to a shot  
PASS  Freeze-frame row counts match ff_n_players  
PASS  Summary totals equal the shots table  
PASS  No missing values in key shot columns  
PASS  is_goal agrees with outcome == 'Goal'  
PASS  statsbomb_xg within [0, 1]  
PASS  Shot locations on the pitch (1-yard tolerance)  (2 strictly outside 0-120 x 0-80)
PASS  Freeze-frame locations on the pitch  
PASS  Shots attack towards x = 120 (median x > 90)  (median x = 104.7)
PASS  Non-penalty goal rate plausible (5-15%)  (0.103)

14/14 checks passed


In [8]:
# A few quick looks at the data
print("Shots per league:")
print(shots_out.groupby("league").agg(matches=("match_id", "nunique"), shots=("shot_id", "size"), goals=("is_goal", "sum")))
print()
print("Shot types:", shots_out["shot_type"].value_counts().to_dict())
print(f"Shots with a freeze frame: {shots_out['has_freeze_frame'].sum():,} of {len(shots_out):,}")
print("Visible players per frame:")
print(shots_out.loc[shots_out["has_freeze_frame"], "ff_n_players"].describe().round(1))

Shots per league:
                matches  shots  goals
league                               
Bundesliga           68   1756    200
La Liga             867  21181   2653
Ligue 1             435  10346   1105
Premier League      418  10837   1082
Serie A             381  10033    955

Shot types: {'Open Play': 50651, 'Free Kick': 2895, 'Penalty': 597, 'Corner': 10}
Shots with a freeze frame: 53,633 of 54,153
Visible players per frame:
count    53633.0
mean        12.8
std          3.6
min          1.0
25%         10.0
50%         13.0
75%         16.0
max         21.0
Name: ff_n_players, dtype: float64


## 1.7 Save the four CSVs to `data/raw`

In [9]:
summary_out.to_csv(RAW / "summary.csv", index=False)
matches_out.to_csv(RAW / "matches.csv", index=False)
shots_out.to_csv(RAW / "shots.csv", index=False)
freeze_frames_out.to_csv(RAW / "freeze_frames.csv", index=False)

for name in ["summary", "matches", "shots", "freeze_frames"]:
    df = pd.read_csv(RAW / f"{name}.csv")
    cells = df.shape[0] * df.shape[1]
    print(f"saved data/raw/{name}.csv  rows={df.shape[0]:,}  cols={df.shape[1]}  cells={cells:,}")

saved data/raw/summary.csv  rows=27  cols=8  cells=216
saved data/raw/matches.csv  rows=2,169  cols=7  cells=15,183
saved data/raw/shots.csv  rows=54,153  cols=30  cells=1,624,590
saved data/raw/freeze_frames.csv  rows=685,059  cols=7  cells=4,795,413


> **Google Sheets limit:** a single spreadsheet holds at most 10 million cells. `shots.csv` and `freeze_frames.csv` fit, but import them into separate spreadsheets (or separate tabs only if the combined cell count stays under 10 million).

---
# Stage 2: model-ready tables

Reads `data/raw/shots.csv` and `data/raw/freeze_frames.csv` and writes:

- **`shot_table.csv`**: one row per shot. Shot context features, the label `is_goal`, and a match-level train/val/test split.
- **`player_table.csv`**: one row per visible player per shot. Per-player features for both the set model and the graph model. Link to `shot_table` with `shot_id`.

The shooter is **not** in the freeze frame (their position is the shot location). The graph model can add the shooter as an extra node built from `shot_table`.

## 2.1 Load and filter
Penalties are dropped (they are nearly all missing a freeze frame, and the other players barely affect a penalty), as are shots with no freeze frame.

In [10]:
SEED = 42
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}

shots = pd.read_csv(RAW / "shots.csv")
ff = pd.read_csv(RAW / "freeze_frames.csv")
print(f"Loaded {len(shots):,} shots and {len(ff):,} freeze-frame rows")

n_start = len(shots)

shots = shots[shots["shot_type"] != "Penalty"]
print(f"Dropped {n_start - len(shots):,} penalties")

n_before = len(shots)
shots = shots[shots["has_freeze_frame"]].copy()
print(f"Dropped {n_before - len(shots):,} shots without a freeze frame")

print(f"Kept {len(shots):,} shots")
ff = ff[ff["shot_id"].isin(shots["shot_id"])].copy()
print(f"Kept {len(ff):,} freeze-frame rows")

Loaded 54,153 shots and 685,059 freeze-frame rows
Dropped 597 penalties
Dropped 0 shots without a freeze frame
Kept 53,556 shots
Kept 684,973 freeze-frame rows


## 2.2 Geometry helpers
Goal centre is (120, 40) and the posts are at y = 36 and y = 44.

In [11]:
def goal_angle(sx, sy):
    """Angle (radians) subtended by the goalposts as seen from the shooter."""
    ax, ay = GOAL_X - sx, POST_LOW - sy
    bx, by = GOAL_X - sx, POST_HIGH - sy
    cross = ax * by - ay * bx
    dot = ax * bx + ay * by
    return np.abs(np.arctan2(cross, dot))


def in_shooting_cone(px, py, sx, sy):
    """True if the point lies inside the triangle shooter - left post - right post."""
    def side(x1, y1, x2, y2, x3, y3):
        return (x1 - x3) * (y2 - y3) - (x2 - x3) * (y1 - y3)

    d1 = side(px, py, sx, sy, GOAL_X, POST_LOW)
    d2 = side(px, py, GOAL_X, POST_LOW, GOAL_X, POST_HIGH)
    d3 = side(px, py, GOAL_X, POST_HIGH, sx, sy)
    has_neg = (d1 < 0) | (d2 < 0) | (d3 < 0)
    has_pos = (d1 > 0) | (d2 > 0) | (d3 > 0)
    return ~(has_neg & has_pos)


POSITION_GROUP = {
    "Goalkeeper": "GK",
    "Center Back": "DEF", "Left Center Back": "DEF", "Right Center Back": "DEF",
    "Left Back": "DEF", "Right Back": "DEF",
    "Left Wing Back": "DEF", "Right Wing Back": "DEF",
    "Center Defensive Midfield": "MID", "Left Defensive Midfield": "MID", "Right Defensive Midfield": "MID",
    "Center Midfield": "MID", "Left Center Midfield": "MID", "Right Center Midfield": "MID",
    "Left Midfield": "MID", "Right Midfield": "MID",
    "Center Attacking Midfield": "MID", "Left Attacking Midfield": "MID", "Right Attacking Midfield": "MID",
    "Center Forward": "FWD", "Left Center Forward": "FWD", "Right Center Forward": "FWD",
    "Secondary Striker": "FWD", "Left Wing": "FWD", "Right Wing": "FWD",
}

## 2.3 `shot_table`
Coordinates are scaled by the pitch size (x / 120, y / 80) and distances by 120, so everything is roughly in 0-1. These are fixed constants rather than statistics fitted to the data, so no information leaks between train and test. Any standardisation that depends on the data (z-scores) should be fitted on the training split only, in the modelling notebook.

In [12]:
shots = shots.reset_index(drop=True)

# Clip the odd location a fraction of a yard outside the pitch.
sx = shots["location_x"].clip(0, PITCH_L)
sy = shots["location_y"].clip(0, PITCH_W)

dist_to_goal = np.hypot(GOAL_X - sx, GOAL_Y - sy)

shot_table = pd.DataFrame({
    # --- identifiers (not features) ---
    "shot_id": shots["shot_id"],
    "match_id": shots["match_id"],
    "league": shots["league"],
    "season": shots["season"],

    # --- label and benchmark (not features) ---
    "is_goal": shots["is_goal"].astype(int),
    "statsbomb_xg": shots["statsbomb_xg"],

    # --- raw geometry in yards (for plots and baselines) ---
    "location_x": sx,
    "location_y": sy,
    "dist_to_goal_yards": dist_to_goal,

    # --- shot context features ---
    "x_norm": sx / PITCH_L,
    "y_norm": sy / PITCH_W,
    "lateral_offset": (sy - GOAL_Y).abs() / GOAL_Y,
    "dist_to_goal_norm": dist_to_goal / PITCH_L,
    "goal_angle": goal_angle(sx, sy),
    "under_pressure": shots["under_pressure"].astype(int),
    "first_time": shots["first_time"].astype(int),

    # --- freeze-frame summary (optional; leave out of a "no freeze frame" ablation) ---
    "n_visible": shots["ff_n_players"],
    "n_teammates_visible": shots["ff_n_teammates"],
    "n_opponents_visible": shots["ff_n_opponents"],
    "keeper_visible": shots["ff_has_keeper"].astype(int),
})

# One-hot encode the categorical shot descriptors.
onehot_cols = []
for col, prefix in [("body_part", "body"), ("shot_type", "type"),
                    ("technique", "tech"), ("play_pattern", "pattern")]:
    dummies = pd.get_dummies(shots[col], prefix=prefix, dtype=int)
    dummies.columns = [c.replace(" ", "_").replace("-", "_") for c in dummies.columns]
    onehot_cols += list(dummies.columns)
    shot_table = pd.concat([shot_table, dummies], axis=1)

print(f"{len(onehot_cols)} one-hot columns: {onehot_cols}")

23 one-hot columns: ['body_Head', 'body_Left_Foot', 'body_Other', 'body_Right_Foot', 'type_Corner', 'type_Free_Kick', 'type_Open_Play', 'tech_Backheel', 'tech_Diving_Header', 'tech_Half_Volley', 'tech_Lob', 'tech_Normal', 'tech_Overhead_Kick', 'tech_Volley', 'pattern_From_Corner', 'pattern_From_Counter', 'pattern_From_Free_Kick', 'pattern_From_Goal_Kick', 'pattern_From_Keeper', 'pattern_From_Kick_Off', 'pattern_From_Throw_In', 'pattern_Other', 'pattern_Regular_Play']


### Train / validation / test split
Shots from the same match are correlated, so the split is done by **match**, not by shot. It is seeded, so both models see exactly the same split.

In [13]:
rng = np.random.default_rng(SEED)
unique_matches = np.sort(shot_table["match_id"].unique())
rng.shuffle(unique_matches)

n = len(unique_matches)
n_train = int(SPLIT_FRACTIONS["train"] * n)
n_val = int(SPLIT_FRACTIONS["val"] * n)

split_of_match = {}
for i, m in enumerate(unique_matches):
    split_of_match[m] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")

shot_table.insert(4, "split", shot_table["match_id"].map(split_of_match))

print(shot_table.groupby("split").agg(
    matches=("match_id", "nunique"), shots=("shot_id", "size"), goal_rate=("is_goal", "mean")
).round(3))

       matches  shots  goal_rate
split                           
test       326   8083      0.100
train     1518  37464      0.103
val        325   8009      0.109


## 2.4 `player_table`
Per-player features. Relative features are measured from the shooter. `in_cone` marks players standing inside the triangle between the shooter and the two posts, and `dist_to_shot_line` is the perpendicular distance to the straight line from the shooter to the goal centre. Rows are ordered by distance to the shooter within each shot.

In [14]:
shooter = pd.DataFrame({"shot_id": shots["shot_id"], "sx": sx, "sy": sy})
players = ff.merge(shooter, on="shot_id", how="inner")

unmapped = set(players["position"].unique()) - set(POSITION_GROUP)
if unmapped:
    raise ValueError(f"Positions missing from POSITION_GROUP: {unmapped}")

px = players["x"].clip(0, PITCH_L)
py = players["y"].clip(0, PITCH_W)

# Perpendicular distance from each player to the shooter -> goal-centre line.
line_dx = GOAL_X - players["sx"]
line_dy = GOAL_Y - players["sy"]
line_len = np.maximum(np.hypot(line_dx, line_dy), 1e-6)
dist_to_line = np.abs(line_dx * (py - players["sy"]) - line_dy * (px - players["sx"])) / line_len

group = players["position"].map(POSITION_GROUP)

player_table = pd.DataFrame({
    "shot_id": players["shot_id"],
    "x_norm": px / PITCH_L,
    "y_norm": py / PITCH_W,
    "dx_to_shooter": (px - players["sx"]) / PITCH_L,
    "dy_to_shooter": (py - players["sy"]) / PITCH_L,
    "dist_to_shooter": np.hypot(px - players["sx"], py - players["sy"]) / PITCH_L,
    "dist_to_goal": np.hypot(GOAL_X - px, GOAL_Y - py) / PITCH_L,
    "dist_to_shot_line": dist_to_line / PITCH_L,
    "in_cone": in_shooting_cone(px, py, players["sx"], players["sy"]).astype(int),
    "teammate": players["teammate"].astype(int),
    "is_keeper": players["is_keeper"].astype(int),
    "pos_GK": (group == "GK").astype(int),
    "pos_DEF": (group == "DEF").astype(int),
    "pos_MID": (group == "MID").astype(int),
    "pos_FWD": (group == "FWD").astype(int),
})

# Stable order: nearest to the shooter first; player_idx counts within each shot.
player_table = player_table.sort_values(["shot_id", "dist_to_shooter"], kind="stable").reset_index(drop=True)
player_table.insert(1, "player_idx", player_table.groupby("shot_id").cumcount())

print(f"player_table: {player_table.shape}")
player_table.head()

player_table: (684973, 16)


,shot_id,player_idx,x_norm,y_norm,dx_to_shooter,dy_to_shooter,dist_to_shooter,dist_to_goal,dist_to_shot_line,in_cone,teammate,is_keeper,pos_GK,pos_DEF,pos_MID,pos_FWD
0,00007fd2-738e-4b27-bb68-7bdeffc77ee7,0,0.850833,0.29750,0.007500,-0.001667,0.007683,0.201186,0.006130,0,0,0,0,1,0,0
1,00007fd2-738e-4b27-bb68-7bdeffc77ee7,1,0.850833,0.31750,0.007500,0.011667,0.013869,0.192493,0.004024,0,0,0,0,1,0,0
2,00007fd2-738e-4b27-bb68-7bdeffc77ee7,2,0.786667,0.38500,-0.056667,0.056667,0.080139,0.226691,0.079881,0,0,0,0,0,1,0
3,00007fd2-738e-4b27-bb68-7bdeffc77ee7,3,0.851667,0.48375,0.008333,0.122500,0.122783,0.148728,0.087888,0,1,0,0,0,1,0
4,00007fd2-738e-4b27-bb68-7bdeffc77ee7,4,0.868333,0.53875,0.025000,0.159167,0.161118,0.134177,0.105009,0,0,0,0,1,0,0


## 2.5 Checks on the processed tables

In [15]:
CHECKS.clear()

feature_cols_shot = (
    ["x_norm", "y_norm", "lateral_offset", "dist_to_goal_norm", "goal_angle", "under_pressure", "first_time"]
    + onehot_cols
)
feature_cols_player = [c for c in player_table.columns if c not in ("shot_id", "player_idx")]

check("shot_id unique in shot_table", shot_table["shot_id"].is_unique)
check("No penalties remain", (shots["shot_type"] != "Penalty").all())
check("No missing values in shot features / label",
      shot_table[feature_cols_shot + ["is_goal", "split"]].notna().all().all())
check("No missing values in player features", player_table[feature_cols_player].notna().all().all())

check("Every shot has at least one player row", set(shot_table["shot_id"]) == set(player_table["shot_id"]))
counts = player_table.groupby("shot_id").size()
check("Players per shot match n_visible",
      counts.reindex(shot_table["shot_id"]).values.tolist() == shot_table["n_visible"].tolist())
check("At most 21 players per shot", counts.max() <= 21, f"(max = {counts.max()})")

check("Scaled coordinates within [0, 1]",
      shot_table[["x_norm", "y_norm"]].stack().between(0, 1).all()
      and player_table[["x_norm", "y_norm"]].stack().between(0, 1).all())
check("goal_angle within (0, pi]", shot_table["goal_angle"].between(0, np.pi).all())
check("Position one-hots sum to 1",
      (player_table[["pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]].sum(axis=1) == 1).all())

match_splits = shot_table.groupby("match_id")["split"].nunique()
check("No match appears in more than one split", (match_splits == 1).all())
check("All three splits non-empty", shot_table["split"].nunique() == 3)

print()
print(f"{sum(ok for _, ok, _ in CHECKS)}/{len(CHECKS)} checks passed")

PASS  shot_id unique in shot_table  
PASS  No penalties remain  
PASS  No missing values in shot features / label  
PASS  No missing values in player features  
PASS  Every shot has at least one player row  
PASS  Players per shot match n_visible  
PASS  At most 21 players per shot  (max = 21)
PASS  Scaled coordinates within [0, 1]  
PASS  goal_angle within (0, pi]  
PASS  Position one-hots sum to 1  
PASS  No match appears in more than one split  
PASS  All three splits non-empty  

12/12 checks passed


## 2.6 Save to `data/processed`

In [16]:
shot_table.to_csv(PROCESSED / "shot_table.csv", index=False, float_format="%.6f")
player_table.to_csv(PROCESSED / "player_table.csv", index=False, float_format="%.6f")

for name, df in [("shot_table", shot_table), ("player_table", player_table)]:
    cells = df.shape[0] * df.shape[1]
    note = "  <- over Google Sheets' 10M-cell limit, import a sample instead" if cells > 10_000_000 else ""
    print(f"saved data/processed/{name}.csv  rows={df.shape[0]:,}  cols={df.shape[1]}  cells={cells:,}{note}")

print()
print("Column groups for the modelling notebook:")
print("  identifiers / split :", ["shot_id", "match_id", "league", "season", "split"])
print("  label               :", "is_goal")
print("  benchmark only      :", "statsbomb_xg")
print("  shot context        :", feature_cols_shot)
print("  freeze-frame summary:", ["n_visible", "n_teammates_visible", "n_opponents_visible", "keeper_visible"], "(optional)")
print("  player features     :", feature_cols_player)

saved data/processed/shot_table.csv  rows=53,556  cols=44  cells=2,356,464
saved data/processed/player_table.csv  rows=684,973  cols=16  cells=10,959,568  <- over Google Sheets' 10M-cell limit, import a sample instead

Column groups for the modelling notebook:
  identifiers / split : ['shot_id', 'match_id', 'league', 'season', 'split']
  label               : is_goal
  benchmark only      : statsbomb_xg
  shot context        : ['x_norm', 'y_norm', 'lateral_offset', 'dist_to_goal_norm', 'goal_angle', 'under_pressure', 'first_time', 'body_Head', 'body_Left_Foot', 'body_Other', 'body_Right_Foot', 'type_Corner', 'type_Free_Kick', 'type_Open_Play', 'tech_Backheel', 'tech_Diving_Header', 'tech_Half_Volley', 'tech_Lob', 'tech_Normal', 'tech_Overhead_Kick', 'tech_Volley', 'pattern_From_Corner', 'pattern_From_Counter', 'pattern_From_Free_Kick', 'pattern_From_Goal_Kick', 'pattern_From_Keeper', 'pattern_From_Kick_Off', 'pattern_From_Throw_In', 'pattern_Other', 'pattern_Regular_Play']
  freeze-fra